


1.   Re-order year numbering from to start from 2000:2024
2.   Remove noisy rows especially last 4 rows






In [1]:
# Install required packages silently
install.packages(
  c("readxl", "tidyquery"),
  quiet = TRUE,
  verbose = FALSE,
  INSTALL_opts = "--no-test-load"
)

In [2]:
library(readxl)
library(tidyr)

##### Load the dataset

In [3]:
df <- read_xlsx(path = "sample_data/GLOBAL_DATAFLOW_2000-2026.xlsx", sheet ="Unicef data")

# Convert column names to lowercase and replace spaces with dashes
colnames(df) <- tolower(gsub(" ", "_", colnames(df)))

head(df, 5)

geographic_area,indicator,sex,2024,2023,2022,2021,2020,2019,2018,⋯,2009,2008,2007,2006,2005,2004,2003,2002,2001,2000
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,⋯,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
Mali,Percentage of adolescents (aged 15-19 years) who had at least one alcoholic drink at any time during the last 12 months,Female,-,-,-,-,-,4.4,-,⋯,-,-,-,-,-,-,-,-,-,-
Mali,Percentage of adolescents (aged 15-19 years) who had at least one alcoholic drink at any time during the last 12 months,Male,-,-,-,-,-,5.5,-,⋯,-,-,-,-,-,-,-,-,-,-
Mali,Percentage of adolescents (aged 15-19 years) who had at least one alcoholic drink at any time during the last 12 months,Total,-,-,-,-,-,5,-,⋯,-,-,-,-,-,-,-,-,-,-
Mali,Antenatal care 1+ visit - percentage of women (aged 15-49 years) attended at least once during pregnancy by skilled health personnel,Total,69.3,-,-,84.2,-,-,79.5,⋯,-,-,-,70.4,-,-,-,-,56.8,-
Mali,Antenatal care 4+ visits - percentage of women (aged 15-49 years) attended at least four times during pregnancy by any provider,Total,51.3,-,-,44.7,-,-,43.3,⋯,-,-,-,35.4,-,-,-,-,29.9,-


In [4]:
# Convert year columns to numeric and replace with zero for nulls
columns_to_cast = as.list(2000:2024)
year_cols = as.character(columns_to_cast)

df[year_cols] <- lapply(df[year_cols], function(x) {
  x <- suppressWarnings(as.numeric(as.character(x)))
  x[is.na(x)] <- 0
  as.integer(x)
})

head(df, 5)


geographic_area,indicator,sex,2024,2023,2022,2021,2020,2019,2018,⋯,2009,2008,2007,2006,2005,2004,2003,2002,2001,2000
<chr>,<chr>,<chr>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,⋯,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>
Mali,Percentage of adolescents (aged 15-19 years) who had at least one alcoholic drink at any time during the last 12 months,Female,0,0,0,0,0,4,0,⋯,0,0,0,0,0,0,0,0,0,0
Mali,Percentage of adolescents (aged 15-19 years) who had at least one alcoholic drink at any time during the last 12 months,Male,0,0,0,0,0,5,0,⋯,0,0,0,0,0,0,0,0,0,0
Mali,Percentage of adolescents (aged 15-19 years) who had at least one alcoholic drink at any time during the last 12 months,Total,0,0,0,0,0,5,0,⋯,0,0,0,0,0,0,0,0,0,0
Mali,Antenatal care 1+ visit - percentage of women (aged 15-49 years) attended at least once during pregnancy by skilled health personnel,Total,69,0,0,84,0,0,79,⋯,0,0,0,70,0,0,0,0,56,0
Mali,Antenatal care 4+ visits - percentage of women (aged 15-49 years) attended at least four times during pregnancy by any provider,Total,51,0,0,44,0,0,43,⋯,0,0,0,35,0,0,0,0,29,0


##### Reshape the data (Pivot from wide to long)


In [5]:
# Identify the columns to keep as identifiers
id_vars <- names(df)[!(names(df) %in% year_cols)]

# Melt the dataframe
melted_df <- df %>%
  pivot_longer(
    cols = `2000`:`2024`,
    names_to = "year",
    values_to = "mncah_total"
  )

# # Ensure Year is numeric for the x-axis
melted_df$year <- as.integer(melted_df$year)

head(melted_df, 5)

geographic_area,indicator,sex,year,mncah_total
<chr>,<chr>,<chr>,<int>,<int>
Mali,Percentage of adolescents (aged 15-19 years) who had at least one alcoholic drink at any time during the last 12 months,Female,2000,0
Mali,Percentage of adolescents (aged 15-19 years) who had at least one alcoholic drink at any time during the last 12 months,Female,2001,0
Mali,Percentage of adolescents (aged 15-19 years) who had at least one alcoholic drink at any time during the last 12 months,Female,2002,0
Mali,Percentage of adolescents (aged 15-19 years) who had at least one alcoholic drink at any time during the last 12 months,Female,2003,0
Mali,Percentage of adolescents (aged 15-19 years) who had at least one alcoholic drink at any time during the last 12 months,Female,2004,0


In [6]:
# Save the DataFrame to a CSV file
write.csv(df, "sample_data/mncah_analysis_ready_data_2000_2025.csv", row.names = FALSE)